# **Bakta annotation anaysis**

In [1]:
from pathlib import Path
from Bio import SeqIO
from collections import Counter
import pandas as pd

### **Структурная аннотация**

In [2]:
project_dir = Path.home() / ("Desktop/bioinformatics/projects/genome_annotation")

bakta_dir = project_dir / "results/structural/bakta"

genome = SeqIO.read(
    bakta_dir / "ecoli.gbff",
    "genbank"
)

counts = Counter(f.type for f in genome.features)

bakta_counts = pd.DataFrame(
    counts.items(),
    columns=["Feature", "Count"]
).sort_values("Count", ascending=False)

display(bakta_counts)

,Feature,Count
1,gene,4658
2,CDS,4315
4,ncRNA,232
6,tRNA,88
3,regulatory,57
5,rRNA,22
8,repeat_region,2
9,rep_origin,2
0,source,1
7,tmRNA,1


In [3]:
# vs NCBI

features = ["gene", "CDS", "tRNA", "rRNA", "ncRNA"]

ncbi_counts = {
    "gene": 4651,
    "CDS": 4318,
    "tRNA": 86,
    "rRNA": 22,
    "ncRNA": 108
}

comparison = pd.DataFrame({
    "Feature": features,
    "NCBI": [ncbi_counts[f] for f in features],
    "Bakta": [counts.get(f, 0) for f in features]
})

comparison["Difference"] = (comparison["Bakta"] - comparison["NCBI"])

display(comparison)

,Feature,NCBI,Bakta,Difference
0,gene,4651,4658,7
1,CDS,4318,4315,-3
2,tRNA,86,88,2
3,rRNA,22,22,0
4,ncRNA,108,232,124


### **Функциональная аннотация**

In [5]:
bakta_cds = [f for f in genome.features if f.type == "CDS"]

products = [f.qualifiers.get("product", ["unknown"])[0] for f in bakta_cds]

hypothetical = sum("hypothetical protein" in p.lower() for p in products)

print("CDS:", len(bakta_cds))
print("Hypothetical proteins:", hypothetical)
print("С функциональным описанием:",len(bakta_cds) - hypothetical)
print("Доля hypothetical:",f"{hypothetical / len(bakta_cds) * 100:.2f}%")

CDS: 4315
Hypothetical proteins: 234
С функциональным описанием: 4081
Доля hypothetical: 5.42%


In [6]:
for feature in bakta_cds[:10]:
    print(
        feature.qualifiers.get("gene", ["-"])[0],
        "|",
        feature.qualifiers.get("product", ["unknown"])[0]
    )

- | hypothetical protein
- | thr operon leader peptide
- | Bifunctional aspartokinase/homoserine dehydrogenase 1
- | Homoserine kinase
- | Threonine synthase
- | hypothetical protein
- | UPF0246 protein GN242_17910
yaaJ | Uncharacterized transporter YaaJ
- | transaldolase B
mogA | Molybdenum cofactor biosynthesis protein MogA


In [7]:
project_dir = (Path.home()/ "Desktop/bioinformatics/projects/genome_annotation")

protein_path = (project_dir/ "results/structural/bakta/ecoli.faa")

proteins = list(SeqIO.parse(protein_path, "fasta"))

print("Количество белков:", len(proteins))
print("Первый ID:", proteins[0].id)
print("Первая последовательность:", proteins[0].seq[:50])

Количество белков: 4315
Первый ID: KOIAIB_00001
Первая последовательность: LFILTATGNMSLCGLKKECLIAASELVTCRE
